In [8]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
data = pd.read_csv("housing.csv")

X = data.drop("median_house_value", axis=1)
y = data["median_house_value"]

print("Dataset shape:", data.shape)
print("Features shape:", X.shape)
print("Target shape:", y.shape)

data.head()

Dataset shape: (20640, 10)
Features shape: (20640, 9)
Target shape: (20640,)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (16512, 9)
Testing data: (4128, 9)


In [11]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]), categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training data:", X_train_processed.shape)
print("Processed testing data:", X_test_processed.shape)

Processed training data: (16512, 13)
Processed testing data: (4128, 13)


In [12]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_processed)
X_test_scaled = scaler_X.transform(X_test_processed)

y_train_scaled = scaler_y.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
).ravel()

poly = PolynomialFeatures(degree=2)

X_poly_train = poly.fit_transform(X_train_scaled)
X_poly_test = poly.transform(X_test_scaled)

print("Original features:", X_train_scaled.shape[1])
print("Polynomial features:", X_poly_train.shape[1])

Original features: 13
Polynomial features: 105


In [13]:
models = {
    "Linear Regression": LinearRegression(),

    "Polynomial Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "SVR": SVR(kernel="rbf")
}

In [14]:
results = {}

for name, model in models.items():

    if name == "Polynomial Regression":
        model.fit(X_poly_train, y_train)
        y_pred = model.predict(X_poly_test)

    elif name == "SVR":
        model.fit(X_train_scaled, y_train_scaled)

        y_pred_scaled = model.predict(X_test_scaled)

        y_pred = scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).ravel()

    elif name == "Linear Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

    else:
        model.fit(X_train_processed, y_train)
        y_pred = model.predict(X_test_processed)

    results[name] = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2 Score": r2_score(y_test, y_pred)
    }

    print(name, "completed")

Linear Regression completed
Polynomial Regression completed
Decision Tree completed
Random Forest completed
SVR completed


In [15]:
results_df = pd.DataFrame(results).T

print("\nRegression Model Comparison")
print("=" * 60)
print(results_df.round(4))

print("\nBest R2 Score:")
print(results_df["R2 Score"].idxmax())

print("\nLowest MAE:")
print(results_df["MAE"].idxmin())

print("\nLowest MSE:")
print(results_df["MSE"].idxmin())


Regression Model Comparison
                              MAE           MSE  R2 Score
Linear Regression      50670.4892  4.908291e+09    0.6254
Polynomial Regression  46391.9892  4.510001e+09    0.6558
Decision Tree          43631.1705  4.779791e+09    0.6352
Random Forest          31636.3496  2.398753e+09    0.8169
SVR                    38098.8696  3.331540e+09    0.7458

Best R2 Score:
Random Forest

Lowest MAE:
Random Forest

Lowest MSE:
Random Forest


In [26]:
print("Regression Model Comparison")
print("=" * 60)

print(results_df)

print("\nBest R2 Score:")
print(results_df["R2 Score"].idxmax())

print("\nLowest MAE:")
print(results_df["MAE"].idxmin())

print("\nLowest MSE:")
print(results_df["MSE"].idxmin())

Regression Model Comparison
                            MAE       MSE  R2 Score
Linear Regression      0.533200  0.555892  0.575788
Polynomial Regression  0.467001  0.464302  0.645682
Decision Tree          0.454679  0.495235  0.622076
Random Forest          0.327543  0.255368  0.805123
SVR                    0.397283  0.354151  0.729740

Best R2 Score:
Random Forest

Lowest MAE:
Random Forest

Lowest MSE:
Random Forest
